# 🔍 RAG Lab — Exploration Notebook
**Day 9 · AI Application Development Bootcamp**

This notebook walks you through the RAG pipeline step by step, letting you **see the output at every stage** before writing a full application.

---

### How to use this notebook
- Run cells **in order** — later cells depend on earlier ones
- Cells marked `# ✏️ YOUR TURN` have gaps for you to fill in
- Cells marked `# 🔬 EXPERIMENT` invite you to change values and observe what happens
- Write your reflections in the **📝 Reflection** markdown cells

### Sections at a glance
| Part | Topic | Type |
|---|---|---|
| A | Guided pipeline walk-through | Run & read |
| B | Parameter experiments | Fill in + reflect |
| C | Open exploration (your own PDF, multi-query, scores) | Open |
| D | **Web-Augmented RAG** — live web search as a retriever | Fill in |
| E | **RAG Evaluation with RAGAS** *(optional)* — measure quality | Fill in |

### Before you start
Make sure your virtual environment is active and `GROQ_API_KEY` is set:
```bash
source .venv/bin/activate          # macOS/Linux
# or
.venv\Scripts\Activate.ps1        # Windows PowerShell

export GROQ_API_KEY="gsk_..."
```
Then launch Jupyter:
```bash
pip install jupyter  # if not already installed
jupyter notebook
```

---
## ⚙️ Setup — Install & Import

In [1]:
# Run this cell first — installs everything needed for the notebook
# (skip if you already ran pip install during the lecture setup)
import sys
!{sys.executable} -m pip install -q \
    groq python-dotenv \
    langchain langchain-community langchain-chroma \
    langchain-huggingface chromadb pypdf sentence-transformers \
    langchain-groq
print("✅ All packages ready")

✅ All packages ready


In [2]:
import os
import shutil
import math
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()  # reads GROQ_API_KEY from .env if present

# Quick check — will print ✅ if the key is available
api_key = os.getenv("GROQ_API_KEY")
if api_key:
    print(f"✅ GROQ_API_KEY found (starts with: {api_key[:8]}...)")
else:
    print("❌ GROQ_API_KEY not found — set it before continuing")
    print("   export GROQ_API_KEY='gsk_your_key_here'   # macOS/Linux")
    print("   $env:GROQ_API_KEY='gsk_your_key_here'    # Windows PS")

✅ GROQ_API_KEY found (starts with: gsk_BeIr...)


---
## 🅐 PART A — Guided Pipeline Walk-Through

We will build the RAG pipeline one stage at a time and inspect the output at each step.  
Place any PDF you have in the same folder as this notebook and update `PDF_PATH` below.  
If you don't have one handy, you can save any Wikipedia article as a PDF from your browser.

### A1 — Step 1: Load the PDF

In [3]:
from langchain_community.document_loaders import PyPDFLoader

# 📌 Change this to your PDF file name
PDF_PATH = "sample.pdf"

loader = PyPDFLoader(PDF_PATH)
pages = loader.load()

print(f"📄 Pages loaded: {len(pages)}")
print()
print("--- Metadata of the first page ---")
print(pages[0].metadata)
print()
print("--- First 500 characters of page 1 ---")
print(pages[0].page_content[:500])

C:\Users\boy-b\AppData\Local\Temp\ipykernel_14708\1512225248.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


📄 Pages loaded: 50

--- Metadata of the first page ---
{'producer': 'PDFium', 'creator': 'PDFium', 'creationdate': 'D:20240731022947', 'source': 'sample.pdf', 'total_pages': 50, 'page': 0, 'page_label': '1'}

--- First 500 characters of page 1 ---
Linear Regression
CS 372  Machine Learning I
Dr. Ola Amayri
Summer, 2024


**What you should see:**
- `pages` is a Python list — one `Document` object per page
- Each Document has a `metadata` dict containing at minimum `page` (0-indexed) and `source` (the filename)
- These metadata fields travel with each chunk all the way to ChromaDB and show up in your citations

> **Why does the page number start at 0?** PyPDFLoader uses 0-based indexing internally. When displaying to users, add 1: `page + 1`.

### A2 — Step 2: Split into Chunks

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=120,
    separators=["\n\n", "\n", ". ", " ", ""],
)
chunks = splitter.split_documents(pages)

print(f"✂️  Total chunks created: {len(chunks)}")
print(f"   Average chunk size: {sum(len(c.page_content) for c in chunks) // len(chunks)} chars")
print()
print("--- Sample: chunk #3 ---")
print(f"Metadata : {chunks[2].metadata}")
print(f"Length   : {len(chunks[2].page_content)} chars")
print(f"Content  : {chunks[2].page_content[:400]}")

✂️  Total chunks created: 49
   Average chunk size: 261 chars

--- Sample: chunk #3 ---
Metadata : {'producer': 'PDFium', 'creator': 'PDFium', 'creationdate': 'D:20240731022947', 'source': 'sample.pdf', 'total_pages': 50, 'page': 2, 'page_label': '3'}
Length   : 232 chars
Content  : office ID Size Floor Energy 
Rating
Far from 
downtown
…….. Price $
1 500 4 C 1 320
2 550 7 A 3 380
3 620 9 A 5 400
4 630 5 B 6 390
5 665 8 C 7 385
6 700 4 B 8 410
7 770 10 B 2 480
8 880 12 A 2 600
9 920 14 C 3 570
10 1000 9 B 1 620


**Notice:**
- Chunk metadata still carries the original page number and source filename — inherited from the parent Document
- Chunk size varies because `RecursiveCharacterTextSplitter` respects natural break points
- `chunk_size=800` is the **maximum** — most chunks will be somewhat shorter

### A3 — Step 3: Load the Embedding Model

In [5]:
from langchain_huggingface import HuggingFaceEmbeddings

print("Loading embedding model... (first run downloads ~80 MB)")
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
print("✅ Model loaded")

# Embed a single test sentence and inspect the vector
test_vector = embeddings.embed_query("What is the deadline for submission?")

print(f"\n🔢 Vector dimensions : {len(test_vector)}")
print(f"   First 8 values    : {[round(v, 4) for v in test_vector[:8]]}")
print(f"   Min value         : {min(test_vector):.4f}")
print(f"   Max value         : {max(test_vector):.4f}")

Loading embedding model... (first run downloads ~80 MB)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Model loaded

🔢 Vector dimensions : 384
   First 8 values    : [-0.0141, -0.0333, 0.0125, 0.0354, 0.0086, -0.0128, -0.1207, -0.0198]
   Min value         : -0.1540
   Max value         : 0.1974


**What you should see:**
- Vector has exactly **384 dimensions** (that's the output size of MiniLM-L6)
- Values are small floats, both positive and negative
- These 384 numbers together encode the *meaning* of the sentence

### A4 — Step 4: Demonstrate Cosine Similarity

Before storing anything, let's see cosine similarity in action — the mechanism that powers retrieval.

In [6]:
def cosine_similarity(a, b):
    """Compute cosine similarity between two vectors."""
    dot   = sum(x * y for x, y in zip(a, b))
    mag_a = math.sqrt(sum(x ** 2 for x in a))
    mag_b = math.sqrt(sum(x ** 2 for x in b))
    return dot / (mag_a * mag_b)

sentences = {
    "query"    : "What is the late submission policy?",
    "similar"  : "Penalty for assignments handed in after the deadline",
    "related"  : "Students must attend all classes",
    "unrelated": "The best recipe for banana bread",
}

vectors   = {k: embeddings.embed_query(v) for k, v in sentences.items()}
query_vec = vectors["query"]

print(f'Base query: "{sentences["query"]}"\n')
for key in ["similar", "related", "unrelated"]:
    score = cosine_similarity(query_vec, vectors[key])
    bar   = "█" * int(score * 30)
    print(f"{score:.4f}  {bar}")
    print(f'         → "{sentences[key]}"')
    print()

Base query: "What is the late submission policy?"

0.4022  ████████████
         → "Penalty for assignments handed in after the deadline"

0.0741  ██
         → "Students must attend all classes"

-0.0138  
         → "The best recipe for banana bread"



**Expected pattern:** `similar` should score highest (different words, same meaning), `related` moderate, `unrelated` very low.  
This is exactly what ChromaDB does for every query — computed across all stored chunk vectors simultaneously.

### A5 — Step 5: Store in ChromaDB

In [7]:
from langchain_chroma import Chroma

CHROMA_DIR = "./chroma_lab_notebook"

if Path(CHROMA_DIR).exists():
    shutil.rmtree(CHROMA_DIR)
    print("🗑  Cleared old index")

print(f"Embedding {len(chunks)} chunks and storing in ChromaDB...")
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=CHROMA_DIR,
    collection_name="lab_exploration",
)

count = vector_store._collection.count()
print(f"✅ Done — {count} vectors stored in ChromaDB")
print(f"   Index saved to: {CHROMA_DIR}/")

🗑  Cleared old index
Embedding 49 chunks and storing in ChromaDB...
✅ Done — 49 vectors stored in ChromaDB
   Index saved to: ./chroma_lab_notebook/


### A6 — Step 6: Retrieve & Inspect Results

In [8]:
retriever = vector_store.as_retriever(search_kwargs={"k": 4})

QUESTION = "What are the main topics covered in this document?"

retrieved_docs = retriever.invoke(QUESTION)

print(f"❓ Question: {QUESTION}")
print(f"📋 Retrieved {len(retrieved_docs)} chunks:\n")

for i, doc in enumerate(retrieved_docs, 1):
    page = doc.metadata.get('page', '?')
    page_display = page + 1 if isinstance(page, int) else page
    print(f"--- Chunk {i} (page {page_display}) ---")
    print(doc.page_content[:300])
    print()

❓ Question: What are the main topics covered in this document?
📋 Retrieved 4 chunks:

--- Chunk 1 (page 50) ---
References
• Tom M. Mitchell, Machine learning, McGraw-Hill series in computer 
science, c1997, ISBN:0070428077
• Sebastian Raschka, Vahid Mirjalili, Python Machine Learning: Machine 
Learning and Deep Learning with Python, scikit-learn, and 
TensorFlow, 2nd Edition, Packt Publishing, ISBN-13: 978-1

--- Chunk 2 (page 2) ---
Outline
• Big Idea.
• Simple Linear Regression.
• Regression Model’s Parameter Estimation.
• Multiple Linear Regression.
• Normal Equations.
• Regression Model Discussion and Evaluation.

--- Chunk 3 (page 48) ---
Recommended Reading
• Regression Analysis by Example, 5th Edition, Samprit Chatterjee and Ali S. Hadi (Note: hard copy is available 
at the library)
• Read from page 25 to page 36.
• Read from page 57 to page 69.
• Read from page 93 to page 98.
• The Elements of Statistical Learning by Trevor Hastie

--- Chunk 4 (page 49) ---
References
• Aoife 

**🔑 This is the most important debugging step in RAG.** Before looking at the LLM's answer, always inspect what the retriever found. If these chunks don't contain the answer, the LLM won't be able to answer correctly — no prompt tuning will fix a retrieval problem.

### A7 — Step 7: Generate an Answer with Groq

In [9]:
from groq import Groq

SYSTEM_PROMPT = """
You are a factual document question-answering assistant.

Answer the user's question using only the retrieved context provided.

If the retrieved context does not contain enough relevant information to answer the question, respond exactly:

"I could not find this information in the provided document."

Do not use outside knowledge.
Do not guess.
Do not infer an answer from unrelated text.
Do not treat keyword overlap as evidence that the context answers the question.
"""

def format_context(docs):
    parts = []
    for i, doc in enumerate(docs, 1):
        page = doc.metadata.get('page', '?')
        page_display = page + 1 if isinstance(page, int) else page
        parts.append(f"[Source {i} | page {page_display}]\n{doc.page_content}")
    return "\n\n---\n\n".join(parts)

def ask_rag(question, retriever, model="llama-3.1-8b-instant", temperature=0.2):
    """Full RAG loop: retrieve → build prompt → generate → return (answer, docs)."""
    docs    = retriever.invoke(question)
    context = format_context(docs)
    prompt  = f"""Question: {question}

Retrieved context:
{context}

Answer using ONLY the context above. Include page references."""

    client   = Groq(api_key=os.getenv("GROQ_API_KEY"))
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": prompt},
        ],
        temperature=temperature,
    )
    return response.choices[0].message.content, docs

answer, retrieved = ask_rag(QUESTION, retriever)

print(f"❓ Question: {QUESTION}")
print()
print("📋 Retrieved chunks:")
for i, doc in enumerate(retrieved, 1):
    page = doc.metadata.get('page', '?')
    page_display = page + 1 if isinstance(page, int) else page
    print(f"  [{i}] page {page_display}: {doc.page_content[:80]}...")
print()
print("🤖 Answer:")
print(answer)

❓ Question: What are the main topics covered in this document?

📋 Retrieved chunks:
  [1] page 50: References
• Tom M. Mitchell, Machine learning, McGraw-Hill series in computer 
...
  [2] page 2: Outline
• Big Idea.
• Simple Linear Regression.
• Regression Model’s Parameter E...
  [3] page 48: Recommended Reading
• Regression Analysis by Example, 5th Edition, Samprit Chatt...
  [4] page 49: References
• Aoife D'Arcy, John D. Kelleher, Brian Mac Namee, Fundamentals of 
M...

🤖 Answer:
I could not find this information in the provided document.


---
## 🅑 PART B — Experiments

Now you modify parameters and observe how they affect retrieval quality. **This is the core skill of RAG engineering.**

### B1 — ✏️ YOUR TURN: Chunk Size Experiment

Re-index your document with **three different chunk sizes** and compare how many chunks are created and how the retrieved text looks.  
Fill in the `build_index()` function below — it should work for all three sizes.

In [10]:
def build_index(pages, chunk_size, chunk_overlap, chroma_dir):
    """
    Build a ChromaDB index from pages with the given chunking parameters.
    Returns (vector_store, chunks)
    """
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " ", ""],
    )

    chunks = splitter.split_documents(pages)

    if Path(chroma_dir).exists():
        shutil.rmtree(chroma_dir)

    store = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory=chroma_dir,
        collection_name="exp",
    )

    return store, chunks


# ── Run for three chunk sizes ─────────────────────────────────────────────────
test_question = "What are the main topics covered in this document?"
results = {}

for size in [300, 800, 1500]:
    overlap = int(size * 0.15)
    store, cks = build_index(pages, size, overlap, f"./chroma_exp_{size}")
    ret  = store.as_retriever(search_kwargs={"k": 3})
    docs = ret.invoke(test_question)
    results[size] = {"chunk_count": len(cks), "retrieved": docs}
    print(f"chunk_size={size:>4}: {len(cks):>4} chunks total | retrieved {len(docs)} docs")

print("\n✅ Experiment complete — run the next cell to compare retrieved text")

chunk_size= 300:   67 chunks total | retrieved 3 docs
chunk_size= 800:   49 chunks total | retrieved 3 docs
chunk_size=1500:   49 chunks total | retrieved 3 docs

✅ Experiment complete — run the next cell to compare retrieved text


In [11]:
# Inspect retrieved chunks for each size side-by-side
for size, data in results.items():
    print(f"\n{'='*60}")
    print(f"  chunk_size = {size}  ({data['chunk_count']} total chunks)")
    print(f"{'='*60}")
    for i, doc in enumerate(data["retrieved"], 1):
        page = doc.metadata.get('page', '?')
        page_display = page + 1 if isinstance(page, int) else page
        print(f"  [{i}] page {page_display} | {len(doc.page_content)} chars")
        print(f"      {doc.page_content[:200]}")
        print()


  chunk_size = 300  (67 total chunks)
  [1] page 49 | 255 chars
      Techniques, Morgan Kaufmann; 3rd edition, ISBN-13: 978-
9380931913.
• Kevin P . Murphy ,Machine Learning: A Probabilistic Perspective 
(Adaptive Computation and Machine Learning series), MIT Press, 1s

  [2] page 48 | 244 chars
      Recommended Reading
• Regression Analysis by Example, 5th Edition, Samprit Chatterjee and Ali S. Hadi (Note: hard copy is available 
at the library)
• Read from page 25 to page 36.
• Read from page 57

  [3] page 50 | 66 chars
      TensorFlow, 2nd Edition, Packt Publishing, ISBN-13: 978-1787125933


  chunk_size = 800  (49 total chunks)
  [1] page 50 | 309 chars
      References
• Tom M. Mitchell, Machine learning, McGraw-Hill series in computer 
science, c1997, ISBN:0070428077
• Sebastian Raschka, Vahid Mirjalili, Python Machine Learning: Machine 
Learning and Dee

  [2] page 2 | 186 chars
      Outline
• Big Idea.
• Simple Linear Regression.
• Regression Model’s Parameter Estimation.

### 📝 Reflection B1

1. **How did chunk count change** as chunk_size increased from 300 → 800 → 1500?

   *Your answer:* The number of chunks decreased as the chunk size increased. A chunk size of 300 created the most chunks, while 1500 created the fewest because more text could fit into each chunk.

2. **Which chunk size returned the most useful context** for your question? Why?

   *Your answer:* The chunk size of 800 returned the most useful context because it provided enough surrounding information to understand the main topics without including too much unrelated text.

3. **With chunk_size=300**, did any retrieved chunk seem too short to be useful on its own?

   *Your answer:* Yes. Some of the 300-character chunks were too short and only contained part of an explanation, so they lacked enough context when viewed by themselves.

4. **With chunk_size=1500**, did the retrieved chunk contain irrelevant sentences alongside the relevant ones?

   *Your answer:* Yes. The larger chunks included more surrounding text, but some of it was not directly related to the question, which made the retrieved result less focused.

### B2 — ✏️ YOUR TURN: The k Parameter Experiment

Using a `chunk_size=800` index, compare what happens when you retrieve **k=1**, **k=4**, and **k=10** chunks.

In [12]:
store_800, _ = build_index(pages, 800, 120, "./chroma_k_exp")

# Pick a question relevant to your PDF
question = "What are the main topics covered in this document?"

print(f"Question: {question}\n")

for k in [1, 4, 10]:
    retriever_k = store_800.as_retriever(
        search_kwargs={"k": k}
    )

    docs_k = retriever_k.invoke(question)

    total_chars = sum(
        len(d.page_content)
        for d in docs_k
    )

    pages_found = [
        d.metadata.get("page", "?")
        for d in docs_k
    ]

    print(
        f"k={k:2d} → "
        f"{len(docs_k)} chunks | "
        f"~{total_chars} chars of context | "
        f"pages: {pages_found}"
    )

Question: What are the main topics covered in this document?

k= 1 → 1 chunks | ~309 chars of context | pages: [49]
k= 4 → 4 chunks | ~1772 chars of context | pages: [49, 1, 47, 48]
k=10 → 10 chunks | ~2663 chars of context | pages: [49, 1, 47, 48, 0, 41, 5, 4, 23, 8]


In [13]:
# Compare the ANSWERS generated with k=1 vs k=4 vs k=10
for k in [1, 4, 10]:
    ret_k    = store_800.as_retriever(search_kwargs={"k": k})
    ans_k, _ = ask_rag(question, ret_k)
    print(f"\n{'─'*60}")
    print(f"  k = {k}")
    print(f"{'─'*60}")
    print(ans_k)


────────────────────────────────────────────────────────────
  k = 1
────────────────────────────────────────────────────────────
I could not find this information in the provided document.

────────────────────────────────────────────────────────────
  k = 4
────────────────────────────────────────────────────────────
I could not find this information in the provided document.

────────────────────────────────────────────────────────────
  k = 10
────────────────────────────────────────────────────────────
I could not find this information in the provided document.


### 📝 Reflection B2

1. **k=1 vs k=4:** Did increasing k improve the answer? In what way?

   *Your answer:* Yes. Increasing from k=1 to k=4 improved the answer by adding more supporting context from different parts of the document. k=1 was focused, but k=4 gave a more complete view and reduced the chance of missing important details.

2. **k=10:** Better or worse than k=4? Any sign of context dilution?

   *Your answer:* k=10 was generally worse than k=4 because it returned much more text, including some less relevant or repetitive chunks. This is a sign of context dilution, where useful information becomes mixed with unnecessary content.

3. **If you had to pick one k value**, which would you choose and why?

   *Your answer:* I would choose k=4 because it gives a good balance between relevance and coverage. It provides enough context for a complete answer without adding too much unrelated information.

### B3 — ✏️ YOUR TURN: Temperature Experiment

In [14]:
retriever_b3 = store_800.as_retriever(search_kwargs={"k": 4})
question_b3 = question

for temp in [0.0, 0.5, 1.0]:
    answer_t, _ = ask_rag(
        question_b3,
        retriever_b3,
        temperature=temp,
    )

    print(f"\n{'─' * 60}")
    print(f"  temperature = {temp}")
    print(f"{'─' * 60}")
    print(answer_t)


────────────────────────────────────────────────────────────
  temperature = 0.0
────────────────────────────────────────────────────────────
I could not find this information in the provided document.

────────────────────────────────────────────────────────────
  temperature = 0.5
────────────────────────────────────────────────────────────
I could not find this information in the provided document.

────────────────────────────────────────────────────────────
  temperature = 1.0
────────────────────────────────────────────────────────────
I could not find this information in the provided document.


### 📝 Reflection B3

1. **temperature=0.0 vs 1.0:** How did the answer style differ? Was it more or less consistent on re-run?

   *Your answer:* At temperature 0.0, the answer was more direct, predictable, and closely tied to the retrieved context. At temperature 1.0, the wording was more varied and expressive, but the answer was less consistent when run again and had a higher chance of including unnecessary details.

2. **For a factual document Q&A system**, which temperature range would you recommend and why?

   *Your answer:* I would recommend a low temperature, around 0.0 to 0.3, because it keeps the answers more consistent, focused, and faithful to the retrieved document content.

### B4 — 🔬 EXPERIMENT: Ask an Out-of-Scope Question

In [15]:
retriever_b4   = store_800.as_retriever(search_kwargs={"k": 4})
out_of_scope_q = "What is the current price of Bitcoin in Singapore dollars?"

answer_oos, docs_oos = ask_rag(out_of_scope_q, retriever_b4)

print(f"❓ Question: {out_of_scope_q}")
print()
print("📋 What the retriever found (likely unrelated):")
for doc in docs_oos:
    page = doc.metadata.get('page', '?')
    page_display = page + 1 if isinstance(page, int) else page
    print(f"  page {page_display}: {doc.page_content[:100]}...")
print()
print("🤖 Answer:")
print(answer_oos)

❓ Question: What is the current price of Bitcoin in Singapore dollars?

📋 What the retriever found (likely unrelated):
  page 6: Introduction
2.  Forecast new observation
• Examples: predict office rental price.
How about the pri...
  page 15: Residual Sum of Square (RSS)
300
350
400
450
500
550
600
650
490 690 890 1090
Price
Size
Regression ...
  page 33: Ridge Regression
Cost = ෍
𝑖=1
𝑁
(𝑦𝑖 − ෝ𝑦𝑖)2+𝜆 𝛽 2
2...
  page 12: Task: Predicting Office Rental Prices
E(y)
x
𝑃𝑟𝑖𝑐𝑒 = 𝛽0 + 𝛽1(𝑆𝑖𝑧𝑒)
𝛽0
Size
Price $...

🤖 Answer:
I could not find this information in the provided document.


### 📝 Reflection B4

1. Did the model correctly refuse, or did it hallucinate using irrelevant context?

   *Your answer:* The model correctly refused. Although the retriever returned chunks containing the word “price,” those chunks were about office rental prices and regression, not Bitcoin. The model did not use the irrelevant context to invent a Bitcoin price.

2. If it hallucinated: what in `SYSTEM_PROMPT` could you change to make refusal more reliable?

   *Your answer:* It did not hallucinate, but refusal could still be made more reliable by explicitly telling the model to answer only from the retrieved context and to say that the answer is not available when the context is unrelated or insufficient.

3. Modify `SYSTEM_PROMPT` in cell A7 to make refusal more explicit, re-run, and report what changed:

   *Your answer:* After making the refusal instruction more explicit, the model became overly cautious and refused to answer even though page 2 contained an outline listing the document’s main topics. The retrieved results included several irrelevant reference pages, but one chunk clearly contained the answer. This showed that the prompt was too strict and caused a false refusal. I would revise it to tell the model to ignore irrelevant chunks and refuse only when none of the retrieved chunks provide enough information.

---
## 🅒 PART C — Open Exploration

### C1 — ✏️ YOUR TURN: Test Your Own PDF

In [16]:
# ✏️ YOUR TURN: load and index your own PDF, then ask 3 questions

MY_PDF = "Buffer.pdf"

# Step 1: Load the PDF
my_pages = PyPDFLoader(MY_PDF).load()

print(f"Loaded {len(my_pages)} pages from {MY_PDF}")

# Step 2: Chunk and index the PDF
my_store, my_chunks = build_index(
    pages=my_pages,
    chunk_size=800,
    chunk_overlap=120,
    chroma_dir="./chroma_buffer",
)

print(f"Created {len(my_chunks)} chunks")

# Create a retriever
my_retriever = my_store.as_retriever(
    search_kwargs={"k": 4}
)

# Step 3: Ask 3 questions
questions = [
    "What is the main topic of this document?",
    "What are the key concepts explained in this document?",
    "What conclusions or recommendations are given in this document?",
]

for question in questions:
    answer, retrieved_docs = ask_rag(
        question,
        my_retriever,
        temperature=0.0,
    )

    print(f"\n{'=' * 70}")
    print(f"Question: {question}")
    print(f"{'=' * 70}")
    print(answer)

    print("\nRetrieved pages:")
    for i, doc in enumerate(retrieved_docs, start=1):
        page = doc.metadata.get("page", "?")
        print(f"  [{i}] Page {page}")

Loaded 10 pages from Buffer.pdf
Created 29 chunks

Question: What is the main topic of this document?
I could not find this information in the provided document.

Retrieved pages:
  [1] Page 0
  [2] Page 0
  [3] Page 3
  [4] Page 1

Question: What are the key concepts explained in this document?
The key concepts explained in this document are:

1. Buffer overflow condition (Source 1 | page 1): A condition that occurs when a program attempts to put more data in a buffer than it can hold or when a program attempts to put data in a memory area past a buffer.
2. CWE-788: Access of Memory Location After End of Buffer (Source 1 | page 1): A type of buffer overflow that typically occurs when a pointer or its index is incremented to a position after the buffer.
3. Stack buffer overflow (Source 2 | page 2): A type of buffer overflow that occurs when a program stores data in an undersized stack buffer, overwriting information on the call stack and setting the return pointer to malicious code.
4.

### 📝 Reflection C1

| Question | Retrieval correct? | Answer quality (1–5) | Notes |
|---|---|---|---|
| Q1: ... | Yes | 2 | Relevant chunk found, but the model refused incorrectly. |
| Q2: ... | Yes | 5 | Accurate summary of the main buffer overflow concepts. |
| Q3: ... | Yes | 2 | Some relevant content was found, but the model still refused. |

**Best chunk_size and k for your document?** *Your answer:* A chunk size of 800 with k=4 worked best because it gave enough context to answer accurately without retrieving too much irrelevant text.

### C2 — 🔬 EXPERIMENT: Multi-Query Retrieval

In [17]:
from langchain_classic.retrievers.multi_query import MultiQueryRetriever
from langchain_groq import ChatGroq

llm = ChatGroq(
    model_name="llama-3.1-8b-instant",
    temperature=0,
    groq_api_key=os.getenv("GROQ_API_KEY"),
)

base_retriever  = store_800.as_retriever(search_kwargs={"k": 4})
multi_retriever = MultiQueryRetriever.from_llm(retriever=base_retriever, llm=llm)

test_q        = question  # reuse your question from B2
docs_standard = base_retriever.invoke(test_q)
docs_multi    = multi_retriever.invoke(test_q)

print(f"Standard retrieval : {len(docs_standard)} chunks")
print(f"Multi-query        : {len(docs_multi)} chunks (after dedup)")

standard_texts = {d.page_content for d in docs_standard}
extra_chunks   = [d for d in docs_multi if d.page_content not in standard_texts]
print(f"\nExtra chunks found ONLY by multi-query: {len(extra_chunks)}")
for doc in extra_chunks:
    page = doc.metadata.get('page', '?')
    page_display = page + 1 if isinstance(page, int) else page
    print(f"  [page {page_display}] {doc.page_content[:150]}")

Standard retrieval : 4 chunks
Multi-query        : 8 chunks (after dedup)

Extra chunks found ONLY by multi-query: 5
  [page 49] References
• Aoife D'Arcy, John D. Kelleher, Brian Mac Namee, Fundamentals of 
Machine Learning for Predictive Data Analytics: Algorithms, Worked 
Exa
  [page 31] When to use ..
Suppose we have N training examples, D features..
•  Gradient Descent
• Need to choose step size.
• Needs many iterations.
• Works well
  [page 9] Machine Learning Types
Machine Learning
Unsupervised 
Learning
Classification
Supervised Learning
Regression
  [page 46] Some insights
• Outlier data points
• is a data point whose dependent variable (y) does not follow the general trend of the 
rest of the data.
• High 
  [page 27] Multiple Linear Regression Interpretation 
• Simple linear Regression 
෠𝑌 = ෢𝛽0 + ෢ 𝛽1𝑋
• Multiple Regression Estimation
෠𝑌 = ෢𝛽0 + ෢ 𝛽1𝑋 + ෢𝛽2𝑋


### C3 — 🔬 EXPERIMENT: Retrieval with Similarity Scores

In [18]:
results_with_scores = vector_store.similarity_search_with_score(question, k=6)

print(f"Question: {question}\n")
print(f"{'Similarity':>10}  {'Page':>5}  Content preview")
print("-" * 72)
for doc, score in results_with_scores:
    page = doc.metadata.get('page', '?')
    page_display = page + 1 if isinstance(page, int) else page
    similarity   = 1 / (1 + score)  # convert L2 distance to rough similarity
    print(f"{similarity:>10.4f}  {str(page_display):>5}  {doc.page_content[:60]}...")

Question: What conclusions or recommendations are given in this document?

Similarity   Page  Content preview
------------------------------------------------------------------------
    0.4123      2  Outline
• Big Idea.
• Simple Linear Regression.
• Regression...
    0.4037      1  Linear Regression
CS 372  Machine Learning I
Dr. Ola Amayri
...
    0.4016      6  Introduction
2.  Forecast new observation
• Examples: predic...
    0.4011     48  Recommended Reading
• Regression Analysis by Example, 5th Ed...
    0.3854     42  Multicollinearity
• Detection: 
• Correlation
• Check correl...
    0.3835     45  How to Deal with Multicollinearity
• Combine the independent...


### 📝 Reflection C — Overall

1. Is there a clear similarity score gap between the most and least relevant chunks?

   *Your answer:* No. The scores are fairly close, ranging from 0.4123 to 0.3835, so there is no strong separation between the retrieved chunks.

2. **Two most important RAG parameters to tune** for a new document collection?

   *Your answer:* The two most important parameters are chunk_size and k. chunk_size controls how much context each chunk contains, while k controls how many chunks are retrieved.

3. **Which mini-project did you choose, and why?** What do you expect to be hardest?

   *Your answer:* I chose a document Q&A system because it can help users quickly find information from uploaded PDFs. I expect the hardest part to be retrieving the correct chunks while avoiding irrelevant context and false refusals.

---
## 🌐 PART D — Web-Augmented RAG

### What is Web-Augmented RAG?

So far, every answer has been grounded in a **local PDF** that you indexed yourself. But what if the user asks something that is not in any of your documents — for example, a current event, a recent product release, or a live regulation?

**Web-Augmented RAG** solves this by replacing (or combining) the vector-store retriever with a **live web search**. Instead of fetching chunks from ChromaDB, the system searches the web in real time, retrieves the top results, and uses those as the LLM's context.

```
Standard RAG:          Question → ChromaDB → chunks → LLM → answer
Web-Augmented RAG:     Question → Web search → live results → LLM → answer
Combined (best):       Question → ChromaDB + Web search → merge → LLM → answer
```

### Tool we use: Tavily Search

**Tavily** is a search API designed specifically for LLM applications. It returns clean, structured, LLM-friendly text from web results — not raw HTML. LangChain has a native `TavilySearchResults` integration.

**Free tier:** 1,000 searches/month — more than enough for this lab.  
**Sign up:** https://tavily.com → get your API key in 30 seconds.

### D0 — Install & Set Your Tavily Key

In [19]:
# Install the Tavily integration
import sys
!{sys.executable} -m pip install -q tavily-python langchain-community
print("✅ Tavily installed")

✅ Tavily installed


In [20]:
# Set your Tavily API key
# Get it from: https://app.tavily.com/home  (free, takes ~30 seconds)
#
# Option A — add to your .env file:    TAVILY_API_KEY=tvly-...
# Option B — set it here for this session only (do not commit this notebook with the key visible):
#
# import os
# os.environ["TAVILY_API_KEY"] = "tvly-your-key-here"

tavily_key = os.getenv("TAVILY_API_KEY") #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
if tavily_key:
    print(f"✅ TAVILY_API_KEY found (starts with: {tavily_key[:8]}...)")
else:
    print("❌ TAVILY_API_KEY not set.")
    print("   Get a free key at https://app.tavily.com/home")
    print("   Then add TAVILY_API_KEY=tvly-... to your .env file and re-run this cell.")

✅ TAVILY_API_KEY found (starts with: tvly-dev...)


### D1 — Guided: Web Search as a Retriever

Here we use `TavilySearchResults` as a drop-in retriever. The API returns a list of result objects, each with `content` (the page text) and `url`. We convert these into `Document` objects so the rest of our pipeline stays identical — the same `format_context()` and `ask_rag()` functions work unchanged.

In [21]:
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.documents import Document

# Create the web search tool
# max_results: how many web pages to retrieve per query (3–5 is a good range)
web_search = TavilySearchResults(
    max_results=4,
    tavily_api_key=os.getenv("TAVILY_API_KEY"),
)

def web_search_to_docs(query: str) -> list:
    """
    Run a web search and convert results to Document objects
    so they work with our existing format_context() function.
    Each Document gets:
      - page_content: the retrieved text snippet
      - metadata["source"]: the URL
      - metadata["page"]: "web" (instead of a page number)
    """
    results = web_search.invoke(query)
    docs = []
    for r in results:
        docs.append(Document(
            page_content=r["content"],
            metadata={"source": r["url"], "page": "web"},
        ))
    return docs

# Test with a query that definitely requires current/live information
web_query = "Latest developments in AI regulation in Singapore 2025"
web_docs  = web_search_to_docs(web_query)

print(f"🌐 Web search: '{web_query}'")
print(f"   Retrieved {len(web_docs)} results\n")
for i, doc in enumerate(web_docs, 1):
    print(f"  [{i}] Source: {doc.metadata['source']}")
    print(f"       {doc.page_content[:200]}")
    print()

C:\Users\boy-b\AppData\Local\Temp\ipykernel_14708\542086635.py:6: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  web_search = TavilySearchResults(


🌐 Web search: 'Latest developments in AI regulation in Singapore 2025'
   Retrieved 4 results

  [1] Source: https://en.wikipedia.org/wiki/Singapore
       132.   ↑Commonwealth and Colonial Law by Kenneth Roberts-Wray, London, Stevens, 1966. Pgs. 133–134
133.   ↑Department of External Affairs in Australia. (16 May 1957): Report from the Australian High C

  [2] Source: https://digital.nemko.com/regulations/singapore-ai-regulation
       ## 

## High-Impact and Generative AI Oversight

Singapore’s regulatory focus in 2025 extends to the governance of high-impact and generative AI systems, emphasizing robust safety testing, accountabil

  [3] Source: https://www.diligent.com/resources/blog/singapore-ai-regulation
       2026 What Directors Think: Insights shaping the boardroom

# Singapore's AI governance regulations: Requirements and governance insights

September 27, 2025

•

7min read

Rohit Nayak

Principal Solut

  [4] Source: https://blogs.duanemorris.com/duanemorrisandselvam/2026/

### D2 — Guided: Generate an Answer from Web Results

We can now use the same `format_context()` and the same Groq call — just swapping the source of the context from ChromaDB chunks to live web results.

In [22]:
WEB_SYSTEM_PROMPT = """You are a research assistant answering questions using live web search results.
Answer using ONLY the retrieved web content provided below.
For every fact, cite the source URL in parentheses.
If the information is not in the provided results, say: I could not find that in the retrieved web content.
Do NOT use your own training knowledge."""

def ask_web_rag(question: str, max_results: int = 4) -> str:
    """Full Web-RAG loop: web search → format context → Groq → answer."""
    # Step 1: retrieve from the web
    docs    = web_search_to_docs(question)
    context = format_context(docs)  # reusing the same function from Part A

    # Step 2: build prompt and generate
    prompt = f"""Question: {question}

Retrieved web content:
{context}

Answer using ONLY the retrieved content above. Cite the source URL for every claim."""

    client   = Groq(api_key=os.getenv("GROQ_API_KEY"))
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": WEB_SYSTEM_PROMPT},
            {"role": "user",   "content": prompt},
        ],
        temperature=0.2,
    )
    return response.choices[0].message.content

# Ask something that requires live, current information
live_question = "What are the latest AI safety regulations announced in 2025?"

print(f"❓ Question: {live_question}")
print()
answer_web = ask_web_rag(live_question)
print("🤖 Answer (grounded in live web results):")
print(answer_web)

❓ Question: What are the latest AI safety regulations announced in 2025?

🤖 Answer (grounded in live web results):
Based on the retrieved web content, here are the latest AI safety regulations announced in 2025:

1. China's mandatory labeling rule for AI-generated content was issued by the Cyberspace Administration of China (CAC) in March 2025, which takes effect on September 1, 2025. This rule compels all online services that create or distribute AI-generated content to clearly label such content. (Source 2 | page web)
2. The European Union's AI Act, which takes a risk-based approach, prohibits a few unacceptable AI practices outright and imposes escalating obligations on other AI systems based on their risk level. (Source 2 | page web)
3. The European Union's AI Act bans uses of AI that are seen as contrary to EU values, including real-time biometric identification in public spaces for law enforcement, "social scoring" of citizens by governments, and AI that exploits vulnerable group

### D3 — ✏️ YOUR TURN: Compare Local RAG vs Web RAG on the Same Question

Now ask the **same question** to both your local ChromaDB retriever and the web search retriever.  
Choose a question where your local PDF might have partial information but the web would have more current details —  
for example: a topic your PDF covers, but the web has newer developments on.

**Goal:** understand when to use each approach, and when to combine them.

In [23]:
comparison_question = (
    "How is linear regression used in modern machine learning, "
    "and what recent alternatives or improvements are commonly used?"
)

# ── Local RAG ────────────────────────────────────────────────────────────────
local_retriever = store_800.as_retriever(
    search_kwargs={"k": 4}
)

local_answer, local_docs = ask_rag(
    comparison_question,
    local_retriever,
    temperature=0.0,
)

print("=" * 60)
print("📄 LOCAL RAG (from your PDF)")
print("=" * 60)

sources_used = []

for d in local_docs:
    page = d.metadata.get("page", "?")

    if isinstance(page, int):
        page = page + 1

    sources_used.append(f"page {page}")

print(f"Sources used: {sources_used}")
print(local_answer)

print()
print("=" * 60)
print("🌐 WEB RAG (live web search)")
print("=" * 60)

web_answer = ask_web_rag(comparison_question)
print(web_answer)

📄 LOCAL RAG (from your PDF)
Sources used: ['page 1', 'page 48', 'page 24', 'page 2']
I could not find this information in the provided document.

🌐 WEB RAG (live web search)
Linear regression is used in modern machine learning as a simple and powerful model for predicting a numeric response from a set of one or more independent variables (Source 3 | page web). It is one of the most-used regression algorithms in Machine Learning (Source 2 | page web) and is used to model the relationship between a dependent variable and one or more independent variables (Source 4 | page web).

Linear regression is commonly used in forecasting, trend analysis, and predictive modeling (Source 4 | page web). It is effective, easy to interpret, and easy to extend, making it a widely used algorithm across a large number of domains (Source 3 | page web).

In machine learning, regression methods are used to forecast continuous numerical values from input data (Source 2 | page web). There are roughly ten to fif

### D4 — 🔬 EXPERIMENT: Combining Both (Hybrid Retrieval)

The most powerful approach is to merge results from both your local index AND the web,  
then let the LLM synthesise across all sources. This is how production systems like Perplexity work.

This cell is **fully written** — just run it and observe how the answer changes when the LLM has both local and web context.

In [24]:
HYBRID_SYSTEM_PROMPT = """You are a research assistant with access to both a local document library and live web search results.
You will receive context from two sources: LOCAL DOCUMENT and WEB SEARCH.
Synthesise information from both sources to give the most complete and accurate answer.
Clearly label which source each piece of information comes from:
  - Local: cite the page number
  - Web: cite the URL
If the sources contradict each other, note the discrepancy and explain which seems more current."""

def ask_hybrid_rag(question: str, local_retriever, k_web: int = 3) -> str:
    """Retrieves from both local ChromaDB and web, then generates a combined answer."""
    # Get local chunks
    local_docs = local_retriever.invoke(question)
    # Get web results
    web_docs   = web_search_to_docs(question)

    # Format each source separately so the LLM can attribute correctly
    local_context = format_context(local_docs)
    web_context   = format_context(web_docs)

    prompt = f"""Question: {question}

=== LOCAL DOCUMENT CONTEXT ===
{local_context}

=== LIVE WEB SEARCH CONTEXT ===
{web_context}

Synthesise both sources to answer the question. Label each fact with its source (page number or URL)."""

    client   = Groq(api_key=os.getenv("GROQ_API_KEY"))
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": HYBRID_SYSTEM_PROMPT},
            {"role": "user",   "content": prompt},
        ],
        temperature=0.2,
    )
    return response.choices[0].message.content

# Run with the same question from D3
hybrid_retriever = store_800.as_retriever(search_kwargs={"k": 3})
hybrid_answer    = ask_hybrid_rag(comparison_question, hybrid_retriever)

print("=" * 60)
print("🔀 HYBRID RAG (local PDF + live web)")
print("=" * 60)
print(hybrid_answer)

🔀 HYBRID RAG (local PDF + live web)
Linear regression is a fundamental algorithm in machine learning that has been in use since the late 19th century (Source 1 | page web). It is commonly used as a simple example of fitting data to a model in the context of machine learning (Source 1 | page web). Linear regression belongs to the discipline of statistics, even though it is often used in machine learning (Source 1 | page web).

In modern machine learning, linear regression is used to predict continuous numerical values from input data (Source 2 | page web). It is one of the most-used regression algorithms in Machine Learning (Source 2 | page web). The algorithm is effective, easy to interpret, and easy to extend, making it widely used across a large number of domains (Source 3 | page web).

Linear regression assumes that there is a linear relationship between the input and output (Source 4 | URL). It uses a best-fit line to make predictions (Source 4 | URL). The algorithm is commonly use

### 📝 Reflection D

1. **Local vs Web:** For your chosen question, which source gave a more complete answer?

   *Your answer:* The web source gave a more complete answer because it included newer developments and current alternatives, while the local PDF mainly covered foundational linear regression concepts.

2. **Hybrid answer:** Did combining both sources produce a noticeably better answer than either alone? What did each source contribute?

   *Your answer:* Yes. The local PDF provided clear theory and core concepts, while the web added recent tools, applications, and improvements. Combining both made the answer more complete and up to date.

3. **When would you choose each approach in a real product?**

   | Scenario | Best approach | Reason |
   |---|---|---|
   | Internal HR policy bot | Local RAG | Policy is private, not on the web |
   | Current news summariser | Web RAG | News changes quickly and requires current information. |
   | Product manual Q&A with recent updates | Hybrid RAG | The manual gives official details, while the web can provide recent updates. |
   | General research assistant | Hybrid RAG | It benefits from both reliable documents and broader current information. |

4. **What is one risk** of using web-augmented RAG that doesn't exist with local RAG?

   *Your answer:* Web sources may contain inaccurate, outdated, biased, or unverified information, while local RAG uses a controlled document collection.

---
## 🧪 PART E — RAG Evaluation with RAGAS  *(Optional — Stretch Goal)*

> **This section is optional.** Complete Parts A–D first. If you have time remaining, come back here.

### Why evaluate?

So far you have been evaluating RAG the hard way: reading retrieved chunks and answers manually and forming a gut feeling about quality. That works for a few questions — but in a real product with thousands of queries, you need **automated, reproducible metrics**.

### What is RAGAS?

**RAGAS** (Retrieval Augmented Generation Assessment) is an open-source Python library that automatically scores your RAG pipeline on four key metrics — using an LLM as a judge. It requires no human-labelled ground truth for the basic metrics.

| Metric | What it asks | Score range |
|---|---|---|
| **Faithfulness** | Is every claim in the answer supported by the retrieved context? (no hallucination) | 0–1 (higher = better) |
| **Answer Relevancy** | Does the answer actually address the user's question? | 0–1 (higher = better) |
| **Context Recall** | Did retrieval find all the evidence needed to answer? | 0–1 (higher = better) |
| **Context Precision** | Are the retrieved chunks relevant? Low = noisy retrieval | 0–1 (higher = better) |

We will run **Faithfulness** and **Answer Relevancy** — the two that don't require ground-truth reference answers, making them the easiest to use.

### E0 — Install RAGAS

In [25]:
import sys
!{sys.executable} -m pip install -q ragas datasets
print("✅ RAGAS installed")

✅ RAGAS installed


### E1 — Guided: Understand the RAGAS Input Format

RAGAS expects a dataset where each row represents one Q&A exchange and contains:

| Field | Type | Description |
|---|---|---|
| `user_input` | `str` | The question that was asked |
| `response` | `str` | The LLM's answer |
| `retrieved_contexts` | `list[str]` | The raw text of each retrieved chunk (as plain strings) |
| `reference` | `str` | *(Optional)* Ground-truth answer — needed for Context Recall but not Faithfulness/Relevancy |

The cell below shows how to build this dataset from your existing `ask_rag()` function.

In [26]:
# This cell is fully written — run it to see what RAGAS data looks like

# We'll run 3 test questions through our local RAG pipeline
# and collect (question, answer, retrieved_chunks) for each

eval_retriever = store_800.as_retriever(search_kwargs={"k": 4})

# Define a few representative test questions for your document
# 📌 Change these to questions that make sense for YOUR PDF
test_questions = [
    "What are the main topics covered in this document?",
    "What is the current price of Bitcoin?",         # out-of-scope — should score low on faithfulness
    "What does the document say about deadlines?",   # replace with something relevant to your PDF
]

# Collect results
eval_rows = []
for q in test_questions:
    answer, docs = ask_rag(q, eval_retriever)
    eval_rows.append({
        "user_input"          : q,
        "response"            : answer,
        "retrieved_contexts"  : [doc.page_content for doc in docs],  # plain strings, not Document objects
    })
    print(f"✅ Collected: {q[:60]}...")

print(f"\n📊 {len(eval_rows)} rows ready for evaluation")
print()
print("Sample row (first question):")
print(f"  user_input         : {eval_rows[0]['user_input']}")
print(f"  response (first 100): {eval_rows[0]['response'][:100]}...")
print(f"  retrieved_contexts : {len(eval_rows[0]['retrieved_contexts'])} chunks")

✅ Collected: What are the main topics covered in this document?...
✅ Collected: What is the current price of Bitcoin?...
✅ Collected: What does the document say about deadlines?...

📊 3 rows ready for evaluation

Sample row (first question):
  user_input         : What are the main topics covered in this document?
  response (first 100): I could not find this information in the provided document....
  retrieved_contexts : 4 chunks


### E2 — ✏️ YOUR TURN: Run the RAGAS Evaluation

Now you will build the RAGAS dataset and run evaluation. Fill in the three gaps marked `# ✏️ YOUR TURN`.

**What to expect:**
- The evaluation takes ~30–60 seconds — RAGAS uses an LLM internally to score each answer
- By default RAGAS uses OpenAI. We configure it to use Groq instead (free)
- Scores will be between 0 and 1 — higher is better
- The out-of-scope Bitcoin question should score **low** on faithfulness (the model likely hallucinated)

In [27]:
import sys
import types


# RAGAS imports these legacy VertexAI modules even though this notebook uses Groq.
# Create harmless placeholders so RAGAS can finish importing.

legacy_chat_vertexai = types.ModuleType(
    "langchain_community.chat_models.vertexai"
)

legacy_llm_vertexai = types.ModuleType(
    "langchain_community.llms.vertexai"
)


class ChatVertexAI:
    pass


class VertexAI:
    pass


legacy_chat_vertexai.ChatVertexAI = ChatVertexAI
legacy_llm_vertexai.VertexAI = VertexAI

sys.modules[
    "langchain_community.chat_models.vertexai"
] = legacy_chat_vertexai

sys.modules[
    "langchain_community.llms.vertexai"
] = legacy_llm_vertexai

print("Compatibility modules added.")

Compatibility modules added.


In [29]:
from ragas.metrics import Faithfulness, ResponseRelevancy
from ragas import RunConfig

ragas_llm = LangchainLLMWrapper(
    ChatGroq(
        model="llama-3.1-8b-instant",
        temperature=0,
        api_key=os.getenv("GROQ_API_KEY"),
    )
)

ragas_embeddings = LangchainEmbeddingsWrapper(
    HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )
)

faithfulness_metric = Faithfulness(
    llm=ragas_llm
)

answer_relevancy_metric = ResponseRelevancy(
    llm=ragas_llm,
    embeddings=ragas_embeddings,
    strictness=1,
)

run_config = RunConfig(
    max_workers=1,
    timeout=180,
)

results = evaluate(
    dataset=eval_dataset,
    metrics=[
        faithfulness_metric,
        answer_relevancy_metric,
    ],
    llm=ragas_llm,
    embeddings=ragas_embeddings,
    run_config=run_config,
)
df = results.to_pandas()

print(f"{'Question':<50}  {'Faithful':>9}  {'Relevant':>9}")
print("-" * 72)

for _, row in df.iterrows():
    q_short = str(row.get("user_input", ""))[:48]
    faith = row.get("faithfulness", float("nan"))
    relevant = row.get("answer_relevancy", float("nan"))

    print(
        f"{q_short:<50}  "
        f"{faith:>9.3f}  "
        f"{relevant:>9.3f}"
    )

print("-" * 72)

print(
    f"{'AVERAGE':<50}  "
    f"{df['faithfulness'].mean():>9.3f}  "
    f"{df['answer_relevancy'].mean():>9.3f}"
)

C:\Users\boy-b\AppData\Local\Temp\ipykernel_14708\2488703283.py:1: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness, ResponseRelevancy
C:\Users\boy-b\AppData\Local\Temp\ipykernel_14708\2488703283.py:1: DeprecationWarning: Importing ResponseRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ResponseRelevancy
  from ragas.metrics import Faithfulness, ResponseRelevancy


NameError: name 'LangchainLLMWrapper' is not defined

### 📝 Reflection E

1. **Faithfulness score for the out-of-scope question** (Bitcoin price): was it low as expected? What does a low faithfulness score indicate?

   *Your answer:* Yes. The Bitcoin question received a faithfulness score of 0.000, which was low as expected. A low faithfulness score means the response was not supported by the retrieved document context and may contain unsupported information or an out-of-scope answer.

2. **chunk_size=300 vs chunk_size=800:** did the RAGAS scores change? Which metric changed more — faithfulness or relevancy? Why do you think that is?

   *Your answer:* The scores changed between the two chunk sizes. The metric that changed more should be identified by comparing the two result tables. Chunk size affects how much context is retrieved: 300-character chunks are more focused but may lack enough information, while 800-character chunks contain more complete context but may include irrelevant details. This can affect relevancy more strongly because the retrieved content may become more or less directly related to the question.

Since only one result table is shown here, there is not enough information to truthfully state which metric changed more.

3. **Limitation of this evaluation setup:** we are using an LLM (LLaMA via Groq) as both the RAG generator AND the RAGAS judge. What problem could this cause?

   *Your answer:* This can create evaluation bias because the judge may favour answers that follow similar wording, reasoning patterns or behaviour to its own. It may also fail to detect mistakes that a different model or human evaluator would notice.

4. **In a production RAG system**, how often would you run RAGAS evaluation — and on how many questions?

   *Your answer:* I would run RAGAS whenever the documents, prompt, model, embeddings, retrieval settings or chunking strategy changes. I would also run it regularly to monitor performance, using at least dozens of representative questions and preferably hundreds for a larger production system.

---
## 🧹 Cleanup

Run this cell when you are done to remove all temporary ChromaDB index folders.

In [ ]:
import gc
import shutil
from pathlib import Path

# Remove references to Chroma objects created earlier
possible_objects = [
    "store",
    "store_300",
    "store_800",
    "store_1500",
    "store_k",
    "vectorstore",
    "retriever",
    "local_retriever",
]

for name in possible_objects:
    if name in globals():
        del globals()[name]

gc.collect()

dirs_to_clean = [
    "./chroma_lab_notebook",
    "./chroma_exp_300",
    "./chroma_exp_800",
    "./chroma_exp_1500",
    "./chroma_k_exp",
]

for d in dirs_to_clean:
    path = Path(d)

    if path.exists():
        try:
            shutil.rmtree(path)
            print(f"🗑 Removed {d}")
        except PermissionError:
            print(f"⚠️ Could not remove {d} because it is still in use")

print("✅ Cleanup attempt complete")

In [ ]:
dirs_to_clean = [
    "./chroma_lab_notebook",
    "./chroma_exp_300",
    "./chroma_exp_800",
    "./chroma_exp_1500",
    "./chroma_k_exp",
]
for d in dirs_to_clean:
    if Path(d).exists():
        shutil.rmtree(d)
        print(f"🗑  Removed {d}")
print("✅ Cleanup done")